# Task 1 - Retail Performance Analytics

Assigned scenario: Marketing Director

Complete the data quality assessment, feature engineering, EDA, pivot/correlation analysis, predictive modelling and visualisation required by the brief. Record your interpretations below each output.

In [24]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [25]:
DATA_DIR = Path('../data')
retail = pd.read_csv(DATA_DIR / 'Retail_regions.csv')
retail.head()

,StoreID,Region,StoreType,MarketingSpend,Footfall,OnlineTraffic,StaffHours,DiscountRate,CustomerSatisfaction,InventoryAvailability,CompetitorDistance,PreviousMonthSales,Season,SalesRevenue
0,1030,South,Retail Park,5216.025748,1360.669726,15143.645217,274.440248,24.246254,84.013149,83.191515,19.065964,46154.901846,Summer,148983.86
1,1013,North,Town,2589.258613,1759.537410,9575.301668,354.439564,33.884441,76.316110,86.923332,15.905459,44674.788107,Autumn,164838.92
2,1022,North,Retail Park,5236.467946,1444.458541,NaN,343.287855,37.862522,79.487759,90.428512,9.477576,48924.722661,Summer,152132.29
3,1094,North,Mall,5848.627748,1369.644357,6489.545492,408.194166,23.574998,63.922735,80.859629,11.220010,52203.354247,Autumn,165743.22
4,1037,North,Retail Park,6647.672155,1073.218324,13003.156557,390.722992,4.795855,58.007873,94.395740,14.487711,33369.483674,Winter,160706.11


In [26]:
# Initial inspection: complete the quality assessment before modelling.
display(retail.shape)
display(retail.dtypes)
display(retail.isna().sum().sort_values(ascending=False))
display(retail.duplicated().sum())
display(retail.describe(include='all').T)

(5100, 14)

StoreID                    int64
Region                       str
StoreType                    str
MarketingSpend           float64
Footfall                 float64
OnlineTraffic            float64
StaffHours               float64
DiscountRate             float64
CustomerSatisfaction     float64
InventoryAvailability    float64
CompetitorDistance       float64
PreviousMonthSales       float64
Season                       str
SalesRevenue             float64
dtype: object

InventoryAvailability    359
MarketingSpend           357
CustomerSatisfaction     356
OnlineTraffic            356
Region                     0
StoreID                    0
Footfall                   0
StoreType                  0
DiscountRate               0
StaffHours                 0
CompetitorDistance         0
PreviousMonthSales         0
Season                     0
SalesRevenue               0
dtype: int64

np.int64(100)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
StoreID,5100.0,NaN,NaN,NaN,1049.660588,28.87401,1000.0,1024.0,1050.0,1075.0,1099.0
Region,5100,4,North,1315,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StoreType,5100,3,Mall,1760,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MarketingSpend,4743.0,NaN,NaN,NaN,5390.11002,4155.709377,250.0,4032.469498,5027.134224,6034.078029,65814.623356
Footfall,5100.0,NaN,NaN,NaN,1203.562926,350.261857,100.0,973.247323,1202.963709,1436.574918,2540.42376
OnlineTraffic,4744.0,NaN,NaN,NaN,8516.080623,2491.154051,500.0,6826.193915,8503.116153,10152.623985,17781.988363
StaffHours,5100.0,NaN,NaN,NaN,348.755755,79.969887,53.587999,294.625585,348.409324,403.279337,651.294406
DiscountRate,5100.0,NaN,NaN,NaN,19.988718,11.529909,0.00315,10.240444,19.835683,30.002817,39.990952
CustomerSatisfaction,4744.0,NaN,NaN,NaN,75.140666,9.909915,41.787013,68.473753,75.238566,82.054071,100.0
InventoryAvailability,4741.0,NaN,NaN,NaN,85.018529,7.778717,54.739076,79.650081,85.111935,90.541148,100.0


## Required quality, feature and summary analysis

Online traffic is filled with its median so that total traffic can be calculated. The three engineered features measure total traffic, sales growth versus the previous month, and revenue generated per unit of marketing spend.

In [27]:
# Investigate outliers using the IQR rule.
numeric_columns = retail.select_dtypes(include='number').columns
outlier_counts = {}

for column in numeric_columns:
    q1 = retail[column].quantile(0.25)
    q3 = retail[column].quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    outlier_counts[column] = ((retail[column] < lower_limit) | (retail[column] > upper_limit)).sum()

display(pd.Series(outlier_counts, name='Outlier count'))

# Handle the missing OnlineTraffic value(s) with the median.
retail['OnlineTraffic'] = retail['OnlineTraffic'].fillna(retail['OnlineTraffic'].median())
    # Three justified engineered features.
retail['TotalTraffic'] = retail['Footfall'] + retail['OnlineTraffic']
retail['SalesGrowthPercent'] = ((retail['SalesRevenue'] - retail['PreviousMonthSales']) / retail['PreviousMonthSales']) * 100
retail['MarketingROI'] = retail['SalesRevenue'] / retail['MarketingSpend']

display(retail[['TotalTraffic', 'SalesGrowthPercent', 'MarketingROI']].head())

StoreID                   0
MarketingSpend           72
Footfall                 44
OnlineTraffic            38
StaffHours               33
DiscountRate              0
CustomerSatisfaction     11
InventoryAvailability    19
CompetitorDistance        0
PreviousMonthSales       54
SalesRevenue             30
Name: Outlier count, dtype: int64

,TotalTraffic,SalesGrowthPercent,MarketingROI
0,16504.314942,222.790980,28.562716
1,11334.839078,268.975270,63.662594
2,9947.574694,210.951768,29.052463
3,7859.189850,217.495346,28.338822
4,14076.374881,381.596034,24.174795


In [28]:
# Four groupby analyses.
region_analysis = retail.groupby('Region').agg(
    AverageSales=('SalesRevenue', 'mean'),
    TotalSales=('SalesRevenue', 'sum'),
    AverageROI=('MarketingROI', 'mean')
).sort_values('AverageSales', ascending=False)

store_type_analysis = retail.groupby('StoreType').agg(
    AverageSales=('SalesRevenue', 'mean'),
    AverageTraffic=('TotalTraffic', 'mean'),
    AverageSatisfaction=('CustomerSatisfaction', 'mean')
).sort_values('AverageSales', ascending=False)

season_analysis = retail.groupby('Season').agg(
    AverageSales=('SalesRevenue', 'mean'),
    AverageDiscount=('DiscountRate', 'mean'),
    AverageGrowth=('SalesGrowthPercent', 'mean')
).sort_values('AverageSales', ascending=False)

region_store_analysis = retail.groupby(['Region', 'StoreType']).agg(
    AverageSales=('SalesRevenue', 'mean'),
    NumberOfStores=('StoreID', 'count')
).sort_values('AverageSales', ascending=False)

display(region_analysis)
display(store_type_analysis)
display(season_analysis)
display(region_store_analysis)

# Two pivot tables.
pivot_region_store = pd.pivot_table(
    retail, values='SalesRevenue', index='Region', columns='StoreType', aggfunc='mean'
)
pivot_season_store = pd.pivot_table(
    retail, values='SalesRevenue', index='Season', columns='StoreType', aggfunc='mean'
)

display(pivot_region_store)
display(pivot_season_store)

,AverageSales,TotalSales,AverageROI
Region,,,
South,155362.864091,1.906302e+08,33.984233
West,153252.297290,1.917186e+08,34.485331
East,149872.124430,1.958829e+08,34.624101
North,149195.858411,1.961926e+08,32.851763


,AverageSales,AverageTraffic,AverageSatisfaction
StoreType,,,
Town,152400.507843,9730.556066,74.911665
Mall,151767.563699,9603.916893,75.181435
Retail Park,151383.604672,9827.582176,75.324055


,AverageSales,AverageDiscount,AverageGrowth
Season,,,
Summer,152459.970947,19.544170,255.436521
Autumn,151728.966779,20.100387,256.529916
Spring,151644.218408,20.147680,249.422877
Winter,151529.866277,20.183832,256.167923


AverageSales  NumberOfStores
Region StoreType                                 
South  Town         155644.071326             430
       Mall         155385.074453             402
       Retail Park  155034.135797             395
West   Mall         153748.161329             414
       Town         153246.665485             412
       Retail Park  152774.726918             425
North  Town         150303.509739             421
East   Town         150254.590025             401
       Mall         150242.113747             467
       Retail Park  149129.177472             439
North  Retail Park  148881.222878             417
       Mall         148493.304549             477

StoreType,Mall,Retail Park,Town
Region,,,
East,150242.113747,149129.177472,150254.590025
North,148493.304549,148881.222878,150303.509739
South,155385.074453,155034.135797,155644.071326
West,153748.161329,152774.726918,153246.665485


StoreType,Mall,Retail Park,Town
Season,,,
Autumn,151719.834559,150284.348505,153173.128054
Spring,151671.437596,151561.994399,151693.337288
Summer,152623.230797,151975.751977,152780.357363
Winter,150980.305376,151657.726968,151965.156765


## Required evidence checklist
- [x] Missing values, duplicates and outlier investigation
- [x] At least three justified engineered features
- [x] Four groupby analyses and two pivot tables
- [ ] Correlation analysis including `SalesRevenue`
- [ ] At least one predictive model with train/test evaluation
- [ ] Feature importance or model interpretation
- [ ] At least three labelled visualisations